# 실험 02: Retriever (검색기)와 의미 기반 검색

## 1. 실험 제목과 목표
- **제목**: Vector DB에서 문서 찾아오기
- **목표**: 저장된 Vector DB에서 사용자의 질문과 가장 유사한 문서를 찾아오는 `Retriever`를 만들고, 키워드 검색과 다른 의미 기반 검색(Semantic Search)의 강력함과 그 한계점을 직접 체험합니다.

## 2. 실험 개요
1. **실험 1 (의미 검색)**: "비밀번호 까먹었어"라는 캐주얼한 질문으로 "계정 암호 재설정 방식"이라는 단어가 전혀 다른 문서를 찾아냅니다.
2. **실험 2 (한계점)**: "휴가 신청은 어떻게 해?"라고 물었을 때, 연차, 병가, 경조휴가 등 의미가 비슷한 수많은 문서를 마구잡이로 가져오는 Vector DB의 단점을 확인합니다.

## 3. 라이브러리 import 및 이전 과정(Vector DB) 복구
이전 노트북에서 했던 과정을 빠르게 반복하여 메모리 DB를 띄웁니다.

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 다양한 사내 규정 문서를 준비합니다 (단어 차이를 눈여겨보세요)
docs = [
    Document(page_content="[IT 규정] 사내 시스템 계정의 암호를 재설정하려면, 포털 우측 상단의 '보안 설정' 메뉴를 이용하세요."),
    Document(page_content="[인사 규정 12조] 연차 신청 절차: 최소 3일 전 사내 ERP를 통해 팀장의 승인을 득해야 합니다."),
    Document(page_content="[인사 규정 15조] 병가 신청 절차: 진단서 첨부 후 ERP 기안, 본부장 전결입니다."),
    Document(page_content="[총무 규정] 식대 청구 방법: 매월 말일 영수증을 스캔하여 총무팀 메일로 발송하세요.")
]

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)

## 4. 실험 1: 의미 기반 검색 (Semantic Search)의 강력함
Retriever(검색기)를 생성하고 질문을 던져봅니다.

In [2]:
# Vector DB를 Retriever(LangChain에서 문서를 찾아오는 표준 인터페이스)로 변환합니다.
# search_kwargs={"k": 1} -> 가장 유사한 문서 1개만 가져오라는 뜻
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

print("=== [키워드가 전혀 다른 캐주얼한 질문] ===")
question = "비밀번호 까먹었어. 어떻게 해?"
print(f"질문: {question}\n")

searched_docs = retriever.invoke(question)

print("[검색된 문서]")
for doc in searched_docs:
    print("->", doc.page_content)
    
print("\n-> 결과: 질문에는 '비밀번호'가 있고 문서에는 '암호'만 있지만, Embedding Model이 두 단어의 의미가 똑같다는 것을 수학적으로 알기 때문에 정확히 찾아냈습니다.")

=== [키워드가 전혀 다른 캐주얼한 질문] ===
질문: 비밀번호 까먹었어. 어떻게 해?

[검색된 문서]
-> [IT 규정] 사내 시스템 계정의 암호를 재설정하려면, 포털 우측 상단의 '보안 설정' 메뉴를 이용하세요.

-> 결과: 질문에는 '비밀번호'가 있고 문서에는 '암호'만 있지만, Embedding Model이 두 단어의 의미가 똑같다는 것을 수학적으로 알기 때문에 정확히 찾아냈습니다.


## 5. 실험 2: 의미 검색의 한계 (문맥이 넓을 때)
너무 포괄적인 질문을 하면 어떻게 될까요?

In [3]:
# 이번엔 3개를 가져오라고 세팅해봅니다.
retriever_top3 = vectorstore.as_retriever(search_kwargs={"k": 3})

print("=== [포괄적인 질문] ===")
question2 = "휴가 신청은 어떻게 해?"
print(f"질문: {question2}\n")

searched_docs2 = retriever_top3.invoke(question2)

print("[검색된 문서 Top 3]")
for i, doc in enumerate(searched_docs2):
    print(f"{i+1}순위 ->", doc.page_content)
    
print("\n-> 주의: '휴가'라는 단어의 의미가 넓기 때문에, 연차와 병가 등 '쉬는 것'과 관련된 문서를 닥치는 대로 다 가져옵니다.")
print("   심지어 관련이 없는 식대 규정도 3순위로 딸려 올 수 있습니다.")
print("   이 문서들이 그대로 프롬프트에 들어가면, LLM은 '휴가를 가려면 진단서를 떼서 영수증과 함께 제출해'라는 끔찍한 환각(Hallucination) 답변을 만들어낼 수 있습니다.")

=== [포괄적인 질문] ===
질문: 휴가 신청은 어떻게 해?

[검색된 문서 Top 3]
1순위 -> [인사 규정 12조] 연차 신청 절차: 최소 3일 전 사내 ERP를 통해 팀장의 승인을 득해야 합니다.
2순위 -> [인사 규정 15조] 병가 신청 절차: 진단서 첨부 후 ERP 기안, 본부장 전결입니다.
3순위 -> [총무 규정] 식대 청구 방법: 매월 말일 영수증을 스캔하여 총무팀 메일로 발송하세요.

-> 주의: '휴가'라는 단어의 의미가 넓기 때문에, 연차와 병가 등 '쉬는 것'과 관련된 문서를 닥치는 대로 다 가져옵니다.
   심지어 관련이 없는 식대 규정도 3순위로 딸려 올 수 있습니다.
   이 문서들이 그대로 프롬프트에 들어가면, LLM은 '휴가를 가려면 진단서를 떼서 영수증과 함께 제출해'라는 끔찍한 환각(Hallucination) 답변을 만들어낼 수 있습니다.


## 6. 결과 해석

1. **유연한 검색**: 과거의 사내 검색 엔진들은 사용자가 정확한 키워드("암호 재설정")를 치지 않으면 아무것도 못 찾았지만, Vector DB는 "까먹었어"라는 말의 의미를 파악합니다.
2. **유사도 랭킹의 함정**: 코사인 유사도라는 단순 수학적 거리만 비교하므로, 정밀도(Precision)가 떨어질 수 있습니다. 즉, 쓸데없는 문서(노이즈)를 많이 가져옵니다.
3. **Retriever의 역할**: `vectorstore.as_retriever()`는 LangChain 체인 안에서 톱니바퀴처럼 부드럽게 맞물려 돌아가게 해주는 필수 어댑터 역할을 합니다.

## 7. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* DB에서 문서를 가져오는(Retrieve) 데 성공함. 단어가 달라도 찰떡같이 알아듣는 게 신기함.
* 하지만 관련 없는 엉뚱한 문서까지 딸려오는 치명적인 부작용을 눈으로 확인함.

### 다음 개선 방향
* 어찌 됐든 문서를 찾아오는 시스템은 완성됨.
* 이제 이 찾아온 문서를 Prompt에 욱여넣고, 최종적으로 LLM이 답변을 생성하게 만드는 **전체 RAG 파이프라인(체인 조립)**을 다음 노트북에서 만들어 보겠음.